# **RQ1**: Number of neighbors per node over time

In [ ]:
# workflow:
# number of neighbors (degree) per node over time (useful for RQ1)
# importare lo snapshot: es. networks\edges_snap1_p99.graphml
# creare un file csv con id dei nodi
# per ogni nodo conto il grado e lo segno nel csv

In [1]:
import pandas as pd
import networkx as nx
import time
from pathlib import Path

# --- CONFIGURAZIONE ---
tsv_path = Path(r"../edges_analysis/edges/edges_snap1_p99.tsv")
output_csv = Path(r"degree_csv/node_degree_snap1_p99.csv")

print(f"[START] Elaborazione TSV: {tsv_path.name}")

t0 = time.time()

# --- 1. Leggi il TSV ---
df = pd.read_csv(tsv_path, sep='\t')
print(f"[INFO] TSV letto in {time.time() - t0:.2f}s - righe: {len(df)}")

# --- 2. Colonne source, target, weight ---
col_source = df.columns[0]
col_target = df.columns[1]
col_weight = df.columns[2] if len(df.columns) > 2 else None

# --- 3. Crea grafo non diretto ---
t1 = time.time()
if col_weight:
    G = nx.from_pandas_edgelist(df, source=col_source, target=col_target, edge_attr=col_weight, create_using=nx.Graph())
else:
    G = nx.from_pandas_edgelist(df, source=col_source, target=col_target, create_using=nx.Graph())

print(f"[INFO] Grafo creato in {time.time() - t1:.2f}s - Nodi: {G.number_of_nodes()}, Archi: {G.number_of_edges()}")

# --- 4. Calcola degree e salva CSV ---
t2 = time.time()
df_degree = pd.DataFrame(G.degree(), columns=["node_id","degree"])
df_degree.to_csv(output_csv, index=False)
print(f"[DONE] Degree CSV salvato in {time.time() - t2:.2f}s")
print(f"[TOTAL] Tempo totale: {time.time() - t0:.2f}s")


[START] Elaborazione TSV: edges_snap1_p99.tsv
[INFO] TSV letto in 2.34s - righe: 12169053
[INFO] Grafo creato in 39.35s - Nodi: 49460, Archi: 12169053
[DONE] Degree CSV salvato in 0.12s
[TOTAL] Tempo totale: 41.86s


In [ ]:
import pandas as pd
import networkx as nx
import time
from pathlib import Path


edges_dir = Path(r"../edges_analysis/edges")
vec_dir = Path(r"../embedding/word_embeddings_cleaned")
output_dir = Path(r"degree_csv")
output_dir.mkdir(exist_ok=True)

percentile = 99  
t0_total = time.time()
print(f"[START] Elaborazione batch snapshot 1-10 (percentile {percentile})")

for snap in range(1, 11):
    t0 = time.time()
    
    edges_tsv = edges_dir / f"edges_snap{snap}_p{percentile}.tsv"
    vec_file = vec_dir / f"fasttext_snap{snap}_filt.vec"
    output_csv = output_dir / f"node_degree_snap{snap}_p{percentile}.csv"
    
    print(f"\n[SNAPSHOT {snap}] Lettura file archi: {edges_tsv.name}")
    
    # archi
    df_edges = pd.read_csv(edges_tsv, sep='\t')
    print(f"[INFO] Righe archi lette: {len(df_edges)}")
    
    # token
    tokens = []
    with open(vec_file, 'r', encoding='utf-8') as f:
        next(f)  # salta la prima riga
        for line in f:
            token = line.strip().split()[0]
            tokens.append(token)
    print(f"[INFO] Token letti: {len(tokens)}")
    
    # Determina colonne source, target, weight 
    col_source = df_edges.columns[0]
    col_target = df_edges.columns[1]
    col_weight = df_edges.columns[2] if len(df_edges.columns) > 2 else None
    
    # Crea grafo non diretto 
    if col_weight:
        G = nx.from_pandas_edgelist(
            df_edges,
            source=col_source,
            target=col_target,
            edge_attr=col_weight,
            create_using=nx.Graph()
        )
    else:
        G = nx.from_pandas_edgelist(
            df_edges,
            source=col_source,
            target=col_target,
            create_using=nx.Graph()
        )
    print(f"[INFO] Grafo creato - Nodi: {G.number_of_nodes()}, Archi: {G.number_of_edges()}")
    
    # Calcola degree e aggiungi token 
    df_degree = pd.DataFrame(G.degree(), columns=["node_id","degree"])
    df_degree["token"] = df_degree["node_id"].apply(lambda x: tokens[x])
    
    # --- 6. Salva CSV ---
    df_degree.to_csv(output_csv, index=False)
    print(f"[DONE] CSV salvato: {output_csv.name} in {time.time() - t0:.2f}s")

print(f"\n[TOTAL] Tempo totale batch: {time.time() - t0_total:.2f}s")


[START] Elaborazione batch snapshot 1-10 (percentile 99)

[SNAPSHOT 1] Lettura file archi: edges_snap1_p99.tsv
[INFO] Righe archi lette: 12169053
[INFO] Token letti: 49460
[INFO] Grafo creato - Nodi: 49460, Archi: 12169053
[DONE] CSV salvato: node_degree_snap1_p99.csv in 133.74s

[SNAPSHOT 2] Lettura file archi: edges_snap2_p99.tsv
[INFO] Righe archi lette: 11035778
[INFO] Token letti: 47189
[INFO] Grafo creato - Nodi: 47189, Archi: 11035778
[DONE] CSV salvato: node_degree_snap2_p99.csv in 209.77s

[SNAPSHOT 3] Lettura file archi: edges_snap3_p99.tsv
[INFO] Righe archi lette: 20314009
[INFO] Token letti: 64187
[INFO] Grafo creato - Nodi: 64187, Archi: 20314009
[DONE] CSV salvato: node_degree_snap3_p99.csv in 207.62s

[SNAPSHOT 4] Lettura file archi: edges_snap4_p99.tsv
[INFO] Righe archi lette: 21038711
[INFO] Token letti: 64631
[INFO] Grafo creato - Nodi: 64631, Archi: 21038711
[DONE] CSV salvato: node_degree_snap4_p99.csv in 202.41s

[SNAPSHOT 5] Lettura file archi: edges_snap5_p99.t